## Transform Customer Data
1. Remove records with NULL customer_id
2. Remove exact duplicate records
3. Remove duplicate records based on created_timestamp
4. CAST the columns to the correct Data Type
5. Write transformed data to the silver schema

1. Remove records with NULL customer_id

In [0]:
%sql
select *
from gizmobox.bronze.v_customers where customer_id is not null
order by customer_id

In [0]:
%sql
select distinct *
from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_customers_distinct AS
select distinct *
from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id

In [0]:
%sql
select customer_id, 
max(created_timestamp)
from v_customers_distinct
group by customer_id

In [0]:
%sql
with cte as (select customer_id, 
max(created_timestamp) as created_timestamp
from v_customers_distinct
group by customer_id)
select v.*
from cte c
join v_customers_distinct v
on c.customer_id = v.customer_id
and c.created_timestamp = v.created_timestamp

In [0]:
%sql
with cte as (select customer_id, 
max(created_timestamp) as created_timestamp
from v_customers_distinct
group by customer_id)
select CAST(v.created_timestamp AS TIMESTAMP) as created_timestamp,
       v.customer_id,
       v.customer_name,
       CAST(v.date_of_birth AS TIMESTAMP) as date_of_birth,
       v.email,
       CAST(v.member_since AS TIMESTAMP) as member_since,
       v.telephone 
from cte c
join v_customers_distinct v
on c.customer_id = v.customer_id
and c.created_timestamp = v.created_timestamp

In [0]:
%sql
CREATE TABLE gizmobox.silver.customers
USING DELTA
LOCATION 'abfss://gizmobox@nagiligaristg.dfs.core.windows.net/silver/customers'
AS
with cte as (select customer_id, 
max(created_timestamp) as created_timestamp
from v_customers_distinct
group by customer_id)
select CAST(v.created_timestamp AS TIMESTAMP) as created_timestamp,
       v.customer_id,
       v.customer_name,
       CAST(v.date_of_birth AS TIMESTAMP) as date_of_birth,
       v.email,
       CAST(v.member_since AS TIMESTAMP) as member_since,
       v.telephone 
from cte c
join v_customers_distinct v
on c.customer_id = v.customer_id
and c.created_timestamp = v.created_timestamp

In [0]:
%sql
select * from gizmobox.silver.customers

In [0]:
%sql
DESCRIBE EXTENDED gizmobox.silver.customers